In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from collections import Counter
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap
from mne.viz import plot_topomap
from scipy.sparse.csgraph import connected_components
from scipy.stats import pearsonr

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from independent_vector_analysis import iva_g  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    analyzers_to_datasets,
    compute_wavelet_datasets,
    load_analyzers,
)
from src.analysis.wavelet_ica import (  # noqa: E402
    align_iva_component_signs,
    zscore_by_time,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# IVA Decomposition of Wavelet Power — Channel Components

## Naming Convention

Notebooks in this directory are named after the **independent dimension** — the feature (mixing) axis where the IVA components live. Here the per-subject mixing axis is the **channel** axis, so each component is a channel topography `(C,)`. The aligned **sample** axis (the SCV that IVA ties across subjects) is the joint **time × frequency** axis, so the *sources* are spectro-temporal — see the companion [`wavelet_iva_frequency_channel.ipynb`](wavelet_iva_frequency_channel.ipynb), whose components instead live on `(frequency × channel)`.

## Motivation

The companion notebook
[`wavelet_iva_frequency_channel.ipynb`](wavelet_iva_frequency_channel.ipynb)
puts `(channel × frequency)` on the **mixing** axis and **time** on the
**sample** axis. Because IVA only ties the *sample* axis across subjects
(the source component vector / SCV), that variant aligns subjects on the
**score timecourse only** — the recovered spatio-spectral patterns are
free per subject, so a shared temporal source is *not* shared in spectrum
or in space.

This notebook keeps the goal — **temporal synchrony between subjects** —
but moves **frequency out of the mixing axis and into the sample axis**:

```
samples  = time × frequency   ← the axis IVA aligns across subjects (SCV)
mixing   = channel            ← free per-subject topography
datasets = subjects
```

Now each IVA component is a **shared-across-subjects spectro-temporal
source** `(F, T)` plus a **per-subject channel topography** `(C,)`. The
SCV ties the *whole time-frequency pattern* across people, so a component
that aligns is aligned in **time and in spectrum jointly** — directly
addressing the "shared temporal pattern is not shared in spectrum" gap.

What this **does not** enforce: shared channel topographies — channel is
the mixing axis, so the scalp maps remain free per subject (analysis (e)
quantifies this as a contrast).

## Reshape

```
Input:   (n_subjects, n_channels, n_freqs, n_times)
Per-subject reshape:
         (n_channels, n_freqs, n_times)
         → (n_channels,  n_freqs × n_times)
           ── mixing ──   ─── samples ───
Per-subject PCA over channels → N_PCA
Stack:   (N_PCA, n_freqs × n_times, n_subjects)  — IVA layout (N, T, K)
```

IVA-G requires a **square** mixing matrix per dataset (`N == feature
dim`), so each subject's `(C, F·T)` matrix is reduced with a **per-subject
PCA over channels** to `N_PCA` components before stacking.

## Why z-scoring matters here

Wavelet power is ~1/f, so without normalisation the low-frequency,
high-power samples would dominate the SCV covariance and the "shared
spectrum" would collapse to "shared low frequencies". `zscore_by_time`
puts every `(subject, channel, frequency)` series at unit variance over
time, which is exactly what makes flattening `F·T` into one sample axis
well-posed.

## Variables produced

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power tensor |
| `X_subjects` | `(S, C, F·T)` | Per-subject z-scored reshape (mixing=C, samples=F·T) |
| `pcas` | list[`PCA`] | Per-subject fitted PCA over channels |
| `X_pca` | `(N_PCA, F·T, S)` | PCA-reduced IVA input layout |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing matrices |
| `iva_scores_pca` | `(S, N_PCA, F·T)` | Per-subject spectro-temporal scores (flat) |
| `iva_sources` | `(S, N_PCA, F, T)` | Spectro-temporal sources reshaped to (freq, time) |
| `iva_time` | `(S, N_PCA, T)` | Frequency-collapsed temporal source (signed) |
| `iva_freq` | `(S, N_PCA, F)` | Time-collapsed spectral source (signed, ≈ 0) |
| `iva_components` | `(S, N_PCA, C)` | Per-subject channel topographies (mixing) |


## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# Choose the experiment: ExperimentNames.PSILO_MUSIC or ExperimentNames.ASSR
EXPERIMENT_NAME = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPES = [MusicTypeVariants.ASSR]
else:
    MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
# Wavelet frequency grid: canonical 1 Hz-spaced grid from src.definitions.frequency
# (WAVELET_FREQ_MIN/MAX/N_FREQS), re-exported via scripts.notebook_helpers.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ─────────────────────────────────────
# NOTE: the sample axis here is F·T, i.e. ``n_freqs`` (=70 broadband) times
# larger than the time-only variants. Keep N_TIMES_SUBSET modest during
# exploration so the (N_PCA, F·T, S) IVA input stays in memory; raise it
# (or set to None) for the full run on a node with more RAM.
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 3000  # first N time samples (F·T = 70 * this)

# ── IVA settings ──────────────────────────────────────────────
# N_COMPONENTS_PCA reduces the *channel* axis here, so it must be
# <= n_channels (<= N_CHANNELS_SUBSET when set). It must also be
# >= N_TOP + N_BOTTOM for the ranking step. ~50 is a sensible target
# for the full 257-channel run; 20 is a fast exploration default.
N_COMPONENTS_PCA = 20  # per-subject PCA dim over channels (= N in IVA's (N, T, K))
IVA_OPT_APPROACH = "newton"  # 'gradient', 'newton', or 'quasi'
IVA_MAX_ITER = 64
IVA_W_DIFF_STOP = 1e-6
IVA_VERBOSE = True
IVA_RANDOM_STATE = 42  # seeds per-subject PCA + W_init

# ── Storage directory ─────────────────────────────────────────
# Reuse the shared stage-03 wavelet cache so the Morlet transform is computed
# once (in notebooks/03-wavelet-analysis) and never recomputed in 04/05.
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "03-wavelet-analysis"
    / "wavelet_cache"
    / EXPERIMENT_NAME.value
)

# ── Downstream analysis settings ──────────────────────────────
# Components are ranked by mean off-diagonal Sigma_N correlation — IVA's own
# estimate of how strongly the kth SCV (the spectro-temporal source) couples
# across subjects. TOP and BOTTOM components are shown separately so the
# bottom set serves as a noise / no-alignment contrast.
N_TOP = 10
N_BOTTOM = 5
N_COMPONENTS_SHOW = N_TOP + N_BOTTOM  # total panels per downstream analysis
SAVE_PLOTS = True
# Plots directory: this notebook's independent (sample / SCV) axis is the
# spectro-temporal one.
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "05-wavelet-iva-analysis"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / "iva_channel"
    / f"pca_{N_COMPONENTS_PCA}"  # isolate sweeps over N_COMPONENTS_PCA
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(f"Plots directory        : {PLOTS_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Per-subject PCA dim    : {N_COMPONENTS_PCA}  (over channels)")
print(f"IVA optimisation       : {IVA_OPT_APPROACH}  (max_iter={IVA_MAX_ITER})")
print(
    f"Components to show     : top={N_TOP} + bottom={N_BOTTOM} "
    f"(total {N_COMPONENTS_SHOW})"
)

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
    experiment_name=EXPERIMENT_NAME,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/` (shared with the 04 ICA notebooks).

In [ ]:
broadband_datasets = compute_wavelet_datasets(
    datasets=datasets,
    analyzers=analyzers,
    experiment_name=EXPERIMENT_NAME,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types. The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

# ── Stimulus onset markers ────────────────────────────────────
# Onsets were extracted during concatenation (identical across subjects by
# construction) and stored on the analyzer as sample indices on the
# concatenated grid. The wavelet time axis shares that grid, so they map
# straight onto ``time``. ``None`` for experiments without stimulus
# annotations (e.g. PSILO_MUSIC) -> no markers are drawn.
_onset_samples = analyzers[LABEL].stimulus_onsets
if _onset_samples is None:
    stimulus_onset_times = np.array([])
else:
    # Keep only onsets inside the (possibly time-subset) plotted window.
    stimulus_onset_times = _onset_samples[_onset_samples < n_times] / sfreq
print(f"Stimulus onsets in window : {len(stimulus_onset_times)}")


def mark_stimulus_onsets(ax):
    """Overlay subtle vertical lines at stimulus onsets on a time axis.

    Deliberately faint (thin, dashed, low alpha) so the markers are present
    without dominating the plot. A no-op when no onsets are available, so it
    is safe to call from every time-axis plot regardless of experiment.
    """
    for onset_t in stimulus_onset_times:
        ax.axvline(onset_t, color="0.2", lw=0.6, ls="--", alpha=0.3, zorder=3)

---
## Stimulus Onset Timeline

A minimal reference plot: one vertical line per stimulus onset along
time. Empty for experiments without stimulus annotations.


In [ ]:
# Simple timeline: one vertical line per stimulus onset.
_all_onsets = analyzers[LABEL].stimulus_onsets
if _all_onsets is None or len(_all_onsets) == 0:
    print(f"No stimulus onsets for {LABEL}; nothing to plot.")
else:
    onset_sec = _all_onsets / sfreq
    fig, ax = plt.subplots(figsize=(14, 1.6))
    ax.vlines(onset_sec, 0, 1, color="steelblue", lw=0.8)
    ax.set_yticks([])
    ax.set_xlim(0, float(onset_sec[-1]) + float(onset_sec[1] - onset_sec[0]))
    ax.set_xlabel("Time (s)")
    ax.set_title(f"Stimulus onsets ({len(onset_sec)}) — {LABEL}")
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "stimulus_onsets_timeline.png", dpi=150,
                    bbox_inches="tight")
    plt.show()
    plt.close("all")

---
## Step 1 — Z-score and Per-Subject Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time series
to zero mean and unit variance (see motivation above — this is what keeps
the `F·T` flatten from collapsing onto low frequencies).

**Reshape** is done independently per subject: each `(C, F, T)` slice
becomes a `(C, F·T)` matrix — **channels** form the mixing axis and the
**joint (frequency, time)** axis forms the samples. The flatten keeps
frequency slow and time fast (`index = f·T + t`), so the sample axis
reshapes cleanly back to `(F, T)` after IVA.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power |
| `X_subjects` | `(S, C, F·T)` | Per-subject z-scored reshape (mixing=C, samples=F·T) |

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Per-subject reshape: (S, C, F, T) → (S, C, F*T). bb_z is C-contiguous, so the
# flattened sample axis runs frequency-slow, time-fast (index = f*T + t) and
# reshapes cleanly back to (F, T) after IVA.
n_samples_ft = n_freqs * n_times
X_subjects = bb_z.reshape(n_subjects, n_channels, n_samples_ft)  # (S, C, F*T)

print(f"Per-subject reshape    : {X_subjects.shape}  (subjects, C, F*T)")
print(f"  Mixing dim (channels): {n_channels}")
print(f"  Samples per subject  : {n_samples_ft}  (F={n_freqs} * T={n_times})")

---
## Step 2 — Per-Subject PCA Over Channels

`iva_g` assumes a **square** mixing matrix per dataset. Here the mixing
axis is **channels**, so each subject's `(C, F·T)` matrix is reduced with
its **own PCA over channels** to `N_PCA` channel-mixtures, after which the
K subject matrices are stacked into IVA's `(N, T, K)` layout.

Note this is the mirror of the frequency_channel notebook: there PCA
reduced the `(C·F)` feature axis with time as samples; here PCA reduces the
channel axis with `(F·T)` as samples. `N_PCA` must therefore be
`<= n_channels`.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pcas[k]` | — | Fitted PCA object for subject `k` (over channels) |
| `pca_evr` | `(S, N_PCA)` | Explained-variance ratio per subject |
| `X_pca` | `(N_PCA, F·T, S)` | IVA input layout (N, T, K) |

In [ ]:
if N_COMPONENTS_PCA > n_channels:
    raise ValueError(
        f"N_COMPONENTS_PCA ({N_COMPONENTS_PCA}) must be <= n_channels "
        f"({n_channels}); PCA reduces the channel axis in this notebook."
    )

# Per-subject PCA over channels. PCA expects (n_samples, n_features); each
# subject's matrix is (C, F*T) → transpose to (F*T, C), fit, transform back to
# (F*T, N_PCA), transpose to (N_PCA, F*T), then stack along axis 2 for IVA.
pcas: list[PCA] = []
pca_scores_per_subject = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
pca_evr = np.zeros((n_subjects, N_COMPONENTS_PCA))

for k in range(n_subjects):
    subj_matrix = X_subjects[k].T  # (F*T, C) — samples x channels for sklearn
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=IVA_RANDOM_STATE)
    scores = pca.fit_transform(subj_matrix)  # (F*T, N_PCA)
    pcas.append(pca)
    pca_scores_per_subject[k] = scores.T  # (N_PCA, F*T)
    pca_evr[k] = pca.explained_variance_ratio_
    print(
        f"  S{k + 1}: explained variance = {pca_evr[k].sum() * 100:5.1f}% "
        f"({N_COMPONENTS_PCA} channel comps)"
    )

# IVA expects (N, T, K)
X_pca = np.ascontiguousarray(pca_scores_per_subject.transpose(1, 2, 0))

print(f"\nIVA input shape        : {X_pca.shape}  (N_PCA, F*T, K=subjects)")

# Quick visual of how much variance each subject's channel PCA retains.
fig, ax = plt.subplots(figsize=(8, 4))
for k in range(n_subjects):
    ax.plot(
        np.arange(1, N_COMPONENTS_PCA + 1),
        np.cumsum(pca_evr[k]),
        marker="o",
        markersize=3,
        label=f"S{k + 1}",
    )
ax.set_xlabel("Number of PCA components (channels)")
ax.set_ylabel("Cumulative variance explained")
ax.set_title(f"Per-Subject Channel PCA — {LABEL}")
ax.legend(fontsize=8, ncol=2)
ax.axhline(0.9, ls="--", lw=0.6, color="gray")
fig.tight_layout()
plt.show()
plt.close("all")

---
## Step 3 — Run IVA-G

`iva_g` returns a demixing matrix `W` of shape `(N, N, K)`. The
spectro-temporal scores for subject `k` are

```
S_pca[k]      = W[:, :, k] @ X_pca[:, :, k]          # (N_PCA, F·T)
components[k] = W[:, :, k] @ pcas[k].components_      # (N_PCA, C)  channel map
```

Because IVA's permutation ambiguity is **shared across datasets**, the
kth source in subject 0 corresponds to the kth source in subjects
`1..K-1` — no post-hoc matching. Here that aligned source is a *whole
spectro-temporal map*, so alignment is in time **and** frequency jointly.

| Returned by `iva_g` | Shape | Description |
|---------------------|-------|-------------|
| `W` | `(N_PCA, N_PCA, S)` | Per-subject demixing matrix |
| `cost` | `(n_iter,)` | IVA cost per iteration |
| `Sigma_N` | `(S, S, N_PCA)` | Per-SCV covariance across subjects (over the F·T axis) |
| `isi` | `float` | Joint ISI (only when ground-truth `A` is given) |

In [ ]:
# Deterministic W initialisation: random matrix per subject seeded by
# IVA_RANDOM_STATE.
rng = np.random.default_rng(IVA_RANDOM_STATE)
W_init = rng.standard_normal((N_COMPONENTS_PCA, N_COMPONENTS_PCA, n_subjects))

W, cost, Sigma_N, isi = iva_g(
    X_pca,
    opt_approach=IVA_OPT_APPROACH,
    whiten=True,
    verbose=IVA_VERBOSE,
    W_init=W_init,
    max_iter=IVA_MAX_ITER,
    W_diff_stop=IVA_W_DIFF_STOP,
)

print(f"\nW shape           : {W.shape}  (N_PCA, N_PCA, K=subjects)")
print(f"Sigma_N shape     : {Sigma_N.shape}  (K, K, N_PCA)")
print(f"Iterations        : {len(cost)}")
print(f"Final cost        : {cost[-1]:.6f}")

# Cost-curve sanity check.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cost, marker="o", markersize=3, color="steelblue")
ax.set_xlabel("Iteration")
ax.set_ylabel("IVA cost")
ax.set_title(f"IVA-G Convergence — {LABEL}")
fig.tight_layout()
plt.show()
plt.close("all")

In [ ]:
np.diag(Sigma_N[:, :, 0])

In [ ]:
# Resolve IVA's per-subject sign ambiguity. IVA recovers each component (SCV)
# only up to a per-subject sign, so subject i's copy of component k may be the
# negation of subject j's — which makes genuinely shared activity look
# anti-correlated. For each component we normalise Sigma_N to a subject x subject
# correlation matrix, take its leading eigenvector (the dominant cross-subject
# direction, oriented so its largest-magnitude entry is positive), and flip every
# subject whose loading on it is negative. Flips are applied to both W and the
# correlation stack (sigma_corr), so every quantity recovered from W below is
# sign-aligned across subjects.
sigma_corr, W, sign_flips = align_iva_component_signs(Sigma_N, W)
n_flipped = int((sign_flips < 0).sum())
print(
    f"Sign alignment: flipped {n_flipped} (component, subject) pairs "
    f"across {N_COMPONENTS_PCA} components."
)
print(f"sigma_corr shape  : {sigma_corr.shape}  (N_PCA, K, K)")
print(f"W shape           : {W.shape}  (N_PCA, N_PCA, K) — sign-aligned")

---
## Step 4 — Recover Spectro-Temporal Sources and Channel Patterns

Apply the per-subject demixing matrix to obtain the IVA scores on the
aligned `F·T` axis, reshape them back to `(F, T)` spectro-temporal
**sources**, and combine `W_k` with the PCA loadings to express each
component's **channel topography** `(C,)`.

We also store two marginals of the source for the downstream synchrony
checks:

- `iva_time` — frequency-collapsed temporal source, signed `(S, N_PCA, T)`
- `iva_freq` — time-collapsed spectral source, signed (≈ 0 by construction) `(S, N_PCA, F)`

The kth source/topography is **already aligned** across subjects.

In [ ]:
iva_scores_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
iva_components = np.zeros((n_subjects, N_COMPONENTS_PCA, n_channels))

for k in range(n_subjects):
    W_k = W[:, :, k]  # (N_PCA, N_PCA)
    X_pca_k = X_pca[:, :, k]  # (N_PCA, F*T)

    # Spectro-temporal scores on the aligned sample axis: (N_PCA, F*T)
    iva_scores_pca[k] = W_k @ X_pca_k

    # Project unmixing rows through the per-subject channel PCA loadings to
    # express each component as a channel topography: (N_PCA, C).
    iva_components[k] = W_k @ pcas[k].components_

# Reshape the aligned F*T axis back to (F, T): the shared spectro-temporal
# source. Flatten was frequency-slow, time-fast, so this is exact.
iva_sources = iva_scores_pca.reshape(
    n_subjects, N_COMPONENTS_PCA, n_freqs, n_times
)  # (S, N_PCA, F, T)

# Signed marginals over the collapsed axis — signs preserved for the diverging
# (RdBu_r) plots. NB: iva_freq (mean over time) is ~0 by construction because
# zscore_by_time zeroes each (s, c, f) series' time-mean; kept signed per request.
iva_time = iva_sources.mean(axis=2)  # (S, N_PCA, T) — frequency-collapsed (signed)
iva_freq = iva_sources.mean(axis=3)  # (S, N_PCA, F) — time-collapsed (signed)

print(f"IVA scores (F*T)       : {iva_scores_pca.shape}  (S, N_PCA, F*T)")
print(f"IVA sources (F, T)     : {iva_sources.shape}  (S, N_PCA, F, T)")
print(f"IVA temporal marginal  : {iva_time.shape}  (S, N_PCA, T)")
print(f"IVA spectral marginal  : {iva_freq.shape}  (S, N_PCA, F)")
print(f"IVA channel patterns   : {iva_components.shape}  (S, N_PCA, C)")

---
## Step 5 — Component Ranking and Selection

We rank components by their **mean off-diagonal correlation in
`Sigma_N`** — IVA's own estimate of how strongly the kth SCV (here the
**spectro-temporal source**) couples across subjects, normalised to a
correlation matrix. Higher = the source the IVA model judges most shared
across subjects in joint time-frequency.

The **top `N_TOP`** and **bottom `N_BOTTOM`** components are carried
through all downstream analyses; the bottom set is a noise contrast.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `sigma_corr` | `(N_PCA, S, S)` | Per-component Sigma_N normalised to a correlation matrix |
| `rank_score` | `(N_PCA,)` | Mean off-diagonal Sigma_N correlation per component |
| `TOP_INDICES` / `BOTTOM_INDICES` | — | Selected component indices |
| `SELECTED_INDICES` | `(N_TOP + N_BOTTOM,)` | TOP followed by BOTTOM |
| `SELECTED_RANK_LABELS` | list[str] | Display labels, e.g. `"TOP 1 (IC 7, r=+0.42)"` |
| `SELECTED_IS_TOP` | list[bool] | True for the first N_TOP entries |

In [ ]:
# ``sigma_corr`` is the sign-aligned (N_PCA, K, K) correlation stack built by
# the sign-alignment cell above. Rank by mean off-diagonal correlation
# (diagonal excluded).
off_diag_mask = ~np.eye(n_subjects, dtype=bool)
rank_score = np.array(
    [sigma_corr[k][off_diag_mask].mean() for k in range(N_COMPONENTS_PCA)]
)

order = np.argsort(rank_score)[::-1]  # descending
TOP_INDICES = order[:N_TOP].tolist()
BOTTOM_INDICES = order[-N_BOTTOM:][::-1].tolist()  # worst first
SELECTED_INDICES = TOP_INDICES + BOTTOM_INDICES
SELECTED_IS_TOP = [True] * N_TOP + [False] * N_BOTTOM


def _ic_title(i: int) -> str:
    """Display label for the i-th selected component."""
    k = SELECTED_INDICES[i]
    tag = f"TOP {i + 1}" if i < N_TOP else f"BOT {i - N_TOP + 1}"
    return f"{tag} (IC {k + 1}, r={rank_score[k]:+.2f})"


SELECTED_RANK_LABELS = [_ic_title(i) for i in range(len(SELECTED_INDICES))]

print("Top components (highest mean off-diagonal Sigma_N correlation):")
for i, k in enumerate(TOP_INDICES):
    print(f"  TOP {i + 1:2d}: IC {k + 1:3d}  r = {rank_score[k]:+.3f}")
print("Bottom components (lowest mean off-diagonal Sigma_N correlation):")
for i, k in enumerate(BOTTOM_INDICES):
    print(f"  BOT {i + 1:2d}: IC {k + 1:3d}  r = {rank_score[k]:+.3f}")

# Summary bar plot: all component scores with top/bottom highlighted.
fig, ax = plt.subplots(figsize=(max(8, 0.35 * N_COMPONENTS_PCA), 4.5))
colors = ["lightgray"] * N_COMPONENTS_PCA
for k in TOP_INDICES:
    colors[k] = "steelblue"
for k in BOTTOM_INDICES:
    colors[k] = "firebrick"
xs = np.arange(N_COMPONENTS_PCA)
ax.bar(xs, rank_score, color=colors)
ax.axhline(0.0, ls="--", lw=0.6, color="gray")
ax.set_xticks(xs)
ax.set_xticklabels([f"{k + 1}" for k in range(N_COMPONENTS_PCA)], fontsize=7)
ax.set_xlabel("IVA component index")
ax.set_ylabel("Mean off-diagonal Sigma_N correlation")
ax.set_title(
    f"Component Ranking by Sigma_N Mean Off-Diagonal Correlation — {LABEL}  "
    f"(top {N_TOP} = blue, bottom {N_BOTTOM} = red)"
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_component_ranking.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Result Summary

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` / `bb_z` | `(S, C, F, T)` | Raw / z-scored wavelet power |
| `X_subjects` | `(S, C, F·T)` | Per-subject reshape (mixing=C, samples=F·T) |
| `pcas` / `pca_evr` | — / `(S, N_PCA)` | Per-subject channel PCA |
| `X_pca` | `(N_PCA, F·T, S)` | IVA input |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing |
| `Sigma_N` | `(S, S, N_PCA)` | Per-SCV covariance across subjects |
| `sigma_corr` | `(N_PCA, S, S)` | Sigma_N normalised to correlation |
| `iva_scores_pca` | `(S, N_PCA, F·T)` | Spectro-temporal scores (flat) |
| `iva_sources` | `(S, N_PCA, F, T)` | Spectro-temporal sources |
| `iva_time` | `(S, N_PCA, T)` | Frequency-collapsed temporal source (signed) |
| `iva_freq` | `(S, N_PCA, F)` | Time-collapsed spectral source (signed, ≈ 0) |
| `iva_components` | `(S, N_PCA, C)` | Per-subject channel topographies |
| `rank_score` / `*_INDICES` | — | Ranking + selection |

The kth source/topography is **aligned across subjects** by
construction — no post-hoc matching is required.

# Downstream Analyses

The IVA solution exposes two kinds of axis per component:

| Variable | Shape | Axis | Role |
|----------|-------|------|------|
| `iva_time` / `iva_freq` | `(S, N_PCA, T)` / `(S, N_PCA, F)` | **temporal** / **spectral** signed marginals | the synchrony read-outs |
| `iva_components` | `(S, N_PCA, C)` | **channel** | the free per-subject mixing pattern |

The subject × subject correlation matrices are kept to the two axes of interest:

- **(b)** ISC of the **temporal** marginal (signed) — do subjects share the time course? (the original goal)
- **(c)** ISC of the **spectral** marginal (signed) — do subjects share the spectrum? **Note:** this marginal is ≈ 0 by construction (z-scoring zeroes each series' time-mean), so it may look like noise; it is kept signed, without power rectification, by request.

Channel — the free mixing axis — is not given its own correlation matrix; its across-subject consistency still shows up in the LOO-ISC summary and the topomap mean/variance below.

Components are visualised as **top `N_TOP`** vs **bottom `N_BOTTOM`** by the Step-5 Sigma_N ranking. Plots split into `_top.png` / `_bottom.png`; bar charts keep both in one figure with a separator. All plots save under `PLOTS_DIR` (`plots/broadband/iva_channel/`).

In [ ]:
# Pearson-r thresholds used for the cluster strip below each ISC matrix.
ISC_CLUSTER_THRESHOLDS = (0.3, 0.5, 0.7)

# Discrete colormap: light gray for singletons (0) + tab10 for groups (1..10).
_GROUP_PALETTE = list(plt.colormaps["tab10"].colors)
_CLUSTER_CMAP = ListedColormap(["#dddddd"] + _GROUP_PALETTE)
_CLUSTER_NORM = BoundaryNorm(
    np.arange(-0.5, len(_GROUP_PALETTE) + 1.5, 1.0), _CLUSTER_CMAP.N
)


def _cluster_grid(
    corr_mat: np.ndarray, thresholds=ISC_CLUSTER_THRESHOLDS
) -> np.ndarray:
    """(T, S) grid of within-row cluster IDs. Singletons → 0, groups → 1, 2, ...."""
    grid = np.zeros((len(thresholds), n_subjects), dtype=int)
    for t_idx, thr in enumerate(thresholds):
        adj = (corr_mat >= thr) & ~np.eye(n_subjects, dtype=bool)
        _, labels = connected_components(adj, directed=False)
        counts = Counter(labels.tolist())
        next_group = 1
        group_map: dict[int, int] = {}
        for s in range(n_subjects):
            lab = int(labels[s])
            if counts[lab] == 1:
                grid[t_idx, s] = 0
            else:
                if lab not in group_map:
                    group_map[lab] = next_group
                    next_group += 1
                grid[t_idx, s] = group_map[lab]
    return grid


def _plot_isc_grid(corr_per_comp, labels, fig_title, file_name):
    """Plot ISC matrices + cluster strips for a group of components."""
    n_show = len(corr_per_comp)
    fig, axes = plt.subplots(
        2,
        n_show,
        figsize=(2.6 * n_show, 5.5),
        gridspec_kw={"height_ratios": [3, 1.2]},
        constrained_layout=True,
    )
    if n_show == 1:
        axes = axes.reshape(2, 1)

    im_corr = None
    for i in range(n_show):
        corr_mat = corr_per_comp[i]
        ax_top = axes[0, i]
        im_corr = ax_top.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
        ax_top.set_xticks(range(n_subjects))
        ax_top.set_yticks(range(n_subjects))
        ax_top.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_top.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_top.set_title(labels[i], fontsize=8)

        ax_bot = axes[1, i]
        grid = _cluster_grid(corr_mat)
        ax_bot.imshow(grid, cmap=_CLUSTER_CMAP, norm=_CLUSTER_NORM, aspect="auto")
        for ti in range(grid.shape[0]):
            for sj in range(grid.shape[1]):
                val = int(grid[ti, sj])
                if val > 0:
                    ax_bot.text(
                        sj,
                        ti,
                        str(val),
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="white",
                        weight="bold",
                    )
        ax_bot.set_xticks(range(n_subjects))
        ax_bot.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_bot.set_yticks(range(len(ISC_CLUSTER_THRESHOLDS)))
        ax_bot.set_yticklabels(
            [f"r≥{thr}" for thr in ISC_CLUSTER_THRESHOLDS], fontsize=8
        )
        if i == 0:
            ax_bot.set_ylabel("Threshold")

    fig.suptitle(fig_title, fontsize=12)
    fig.colorbar(im_corr, ax=axes[0, -1], label="Pearson r", shrink=0.8)
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def _isc_stack(axis_data, indices):
    """Subject x subject Pearson corr per component over the last axis.

    ``axis_data`` is (S, N_PCA, L); returns (len(indices), S, S).
    """
    return np.stack([np.corrcoef(axis_data[:, k, :]) for k in indices])

---
## Analysis (a) — Channel PCA Scree

Explained variance of the per-subject **channel** PCA that feeds IVA.
Left: across-subject mean `explained_variance_ratio_` (± std). Right:
per-subject cumulative variance (gray) + across-subject mean (coral).

In [ ]:
# Across-subject summary of per-subject PCA explained-variance ratios.
pca_evr_mean = pca_evr.mean(axis=0)  # (N_PCA,)
pca_evr_std = pca_evr.std(axis=0)  # (N_PCA,)
pca_evr_cum_mean = np.cumsum(pca_evr_mean)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

xs = np.arange(1, N_COMPONENTS_PCA + 1)
axes[0].bar(xs, pca_evr_mean, yerr=pca_evr_std, color="steelblue", capsize=2)
axes[0].set_xlabel("PCA component (channels)")
axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title(f"Channel PCA Scree (mean ± std across subjects) — {LABEL}")

for k in range(n_subjects):
    axes[1].plot(xs, np.cumsum(pca_evr[k]), lw=0.6, alpha=0.5, color="gray")
axes[1].plot(xs, pca_evr_cum_mean, "o-", color="coral", label="mean cumulative")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance — {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"All {N_COMPONENTS_PCA} channel PCA components explain "
    f"{pca_evr_cum_mean[-1] * 100:.1f}% of variance on average."
)

---
## Analysis (b) — ISC of the Temporal Marginal

The original goal: do subjects share the **time course**? Per-subject
vector is the frequency-collapsed (signed) temporal source `iva_time[s, k, :]` (length `T`).
This is the temporal-synchrony read-out, now derived from a source that
was aligned in joint time-frequency rather than in time alone.

In [ ]:
# Subject × subject correlation of the frequency-collapsed temporal source.
_plot_isc_grid(
    _isc_stack(iva_time, TOP_INDICES),
    labels=SELECTED_RANK_LABELS[:N_TOP],
    fig_title=(
        f"ISC of Temporal Marginal (freq-collapsed) — TOP {N_TOP} — {LABEL}"
    ),
    file_name="iva_isc_temporal_top.png",
)
_plot_isc_grid(
    _isc_stack(iva_time, BOTTOM_INDICES),
    labels=SELECTED_RANK_LABELS[N_TOP:],
    fig_title=(
        f"ISC of Temporal Marginal (freq-collapsed) — BOTTOM {N_BOTTOM} — {LABEL}"
    ),
    file_name="iva_isc_temporal_bottom.png",
)

---
## Analysis (c) — ISC of the Spectral Marginal

The new question this notebook is built to answer: do subjects share the
**spectrum**? Per-subject vector is the time-collapsed (signed) spectral source
`iva_freq[s, k, :]` (length `F`). **Caveat:** this marginal is ≈ 0 by
construction — `zscore_by_time` zeroes each `(s, c, f)` series' time-mean, so
the signed mean over time nearly cancels. It is kept signed (no power rectification) by request, so this matrix may look like noise.

In [ ]:
# Subject × subject correlation of the time-collapsed (signed) spectral source.
_plot_isc_grid(
    _isc_stack(iva_freq, TOP_INDICES),
    labels=SELECTED_RANK_LABELS[:N_TOP],
    fig_title=(
        f"ISC of Spectral Marginal (time-collapsed) — TOP {N_TOP} — {LABEL}"
    ),
    file_name="iva_isc_spectral_top.png",
)
_plot_isc_grid(
    _isc_stack(iva_freq, BOTTOM_INDICES),
    labels=SELECTED_RANK_LABELS[N_TOP:],
    fig_title=(
        f"ISC of Spectral Marginal (time-collapsed) — BOTTOM {N_BOTTOM} — {LABEL}"
    ),
    file_name="iva_isc_spectral_bottom.png",
)

---
## Analysis (d) — LOO-ISC Across All Four Axes

Whole-recording leave-one-out ISC per component along each axis
(spectro-temporal `F·T`, temporal `T`, spectral `F`, channel `C`). For
each axis and component `k`:

```
for subject s:
    others = mean over s' ≠ s of vec[s']
    r[k, s] = pearson(vec[s], others)
```

One bar figure per axis (mean ± std across subjects, red where negative,
top|bottom separator), then a **grouped summary** putting the four axes
side by side per top component — the single plot that answers "is this
source shared in time *and* spectrum *and* space?".

In [ ]:
def _loo_isc(axis_data, indices):
    """LOO-ISC per component along the last axis of (S, N_PCA, L).

    Returns (len(indices), S).
    """
    out = np.zeros((len(indices), n_subjects))
    for i, k in enumerate(indices):
        vecs = axis_data[:, k, :]  # (S, L)
        for s in range(n_subjects):
            others_mean = np.delete(vecs, s, axis=0).mean(axis=0)
            out[i, s] = float(pearsonr(vecs[s], others_mean)[0])
    return out


def _plot_loo_bar(axis_data, axis_name, file_name):
    loo = _loo_isc(axis_data, SELECTED_INDICES)  # (N_SHOW, S)
    loo_mean = loo.mean(axis=1)
    loo_std = loo.std(axis=1)
    bar_colors = []
    for i, m in enumerate(loo_mean):
        if SELECTED_IS_TOP[i]:
            bar_colors.append("firebrick" if m < 0 else "steelblue")
        else:
            bar_colors.append("salmon" if m < 0 else "lightsteelblue")

    fig, ax = plt.subplots(figsize=(max(10, 0.9 * N_COMPONENTS_SHOW), 5.0))
    xs = np.arange(N_COMPONENTS_SHOW)
    ax.bar(xs, loo_mean, yerr=loo_std, color=bar_colors, capsize=4)
    ax.axhline(0.0, ls="--", lw=0.6, color="gray")
    ax.axvline(
        N_TOP - 0.5, ls="--", lw=0.9, color="black", alpha=0.5,
        label=f"top {N_TOP} | bottom {N_BOTTOM}",
    )
    ax.set_xticks(xs)
    ax.set_xticklabels(SELECTED_RANK_LABELS, fontsize=7, rotation=45, ha="right")
    ax.set_xlabel("Component")
    ax.set_ylabel("Mean LOO-ISC across subjects")
    ax.set_ylim(-1.05, 1.05)
    ax.set_title(f"Per-IC Mean LOO-ISC — {axis_name} — {LABEL}")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")
    return loo_mean


LOO_AXES = [
    (iva_scores_pca, "Spectro-Temporal (F·T)", "iva_loo_isc_spectrotemporal.png"),
    (iva_time, "Temporal (T)", "iva_loo_isc_temporal.png"),
    (iva_freq, "Spectral (F)", "iva_loo_isc_spectral.png"),
    (iva_components, "Channel (C)", "iva_loo_isc_channel.png"),
]
loo_means = {name: _plot_loo_bar(data, name, fn) for data, name, fn in LOO_AXES}

# Grouped summary: the four axes side by side for the TOP components.
fig, ax = plt.subplots(figsize=(max(10, 1.1 * N_TOP), 5.0))
axis_names = list(loo_means.keys())
width = 0.2
xs = np.arange(N_TOP)
palette = ["steelblue", "darkorange", "seagreen", "mediumpurple"]
for j, name in enumerate(axis_names):
    ax.bar(
        xs + (j - 1.5) * width,
        loo_means[name][:N_TOP],
        width=width,
        color=palette[j],
        label=name,
    )
ax.axhline(0.0, ls="--", lw=0.6, color="gray")
ax.set_xticks(xs)
ax.set_xticklabels(SELECTED_RANK_LABELS[:N_TOP], fontsize=7, rotation=45, ha="right")
ax.set_ylabel("Mean LOO-ISC across subjects")
ax.set_ylim(-1.05, 1.05)
ax.set_title(f"LOO-ISC by Axis — TOP {N_TOP} components — {LABEL}")
ax.legend(loc="upper right", fontsize=8, ncol=2)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_loo_isc_axis_summary.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (e) — Topomap of Mean and Variance Across Subjects

The component pattern *is* a channel topography here (no frequency axis to
collapse): `iva_components[s, k, :]`. Per component we show the
across-subject **mean** spatial map (`RdBu_r`) and **variance**
(`viridis`). High variance flags that the spatial expression of an
otherwise spectro-temporally-shared source differs across people.

In [ ]:
# Channel topography per subject IS the component pattern: (S, N_PCA, C).
chan_loading = iva_components
chan_mean = chan_loading.mean(axis=0)  # (N_PCA, C)
chan_var = chan_loading.var(axis=0)  # (N_PCA, C)

# MNE info for the topomap, restricted to the channel subset.
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))


def _plot_topomap_group(indices, labels, group_name, file_name):
    n_show = len(indices)
    fig, axes = plt.subplots(2, n_show, figsize=(3.0 * n_show, 7.0))
    if n_show == 1:
        axes = axes.reshape(2, 1)
    for i, k in enumerate(indices):
        vlim_m = float(np.percentile(np.abs(chan_mean[k]), 99))
        if vlim_m == 0.0:
            vlim_m = 1e-12
        im_m, _ = plot_topomap(
            chan_mean[k], info, axes=axes[0, i], show=False,
            cmap="RdBu_r", vlim=(-vlim_m, vlim_m),
        )
        axes[0, i].set_title(labels[i], fontsize=8)
        fig.colorbar(im_m, ax=axes[0, i], fraction=0.046, pad=0.04)

        vlim_v = float(np.percentile(chan_var[k], 99))
        if vlim_v == 0.0:
            vlim_v = 1e-12
        im_v, _ = plot_topomap(
            chan_var[k], info, axes=axes[1, i], show=False,
            cmap="viridis", vlim=(0.0, vlim_v),
        )
        fig.colorbar(im_v, ax=axes[1, i], fraction=0.046, pad=0.04)

    fig.text(
        0.01, 0.75, "Mean across subjects", rotation=90, va="center",
        fontsize=11, fontweight="bold",
    )
    fig.text(
        0.01, 0.25, "Variance across subjects", rotation=90, va="center",
        fontsize=11, fontweight="bold",
    )
    fig.suptitle(
        f"Mean and Variance Topomaps Across Subjects — {group_name} — {LABEL}",
        fontsize=13,
    )
    fig.tight_layout(rect=(0.03, 0, 1, 0.97))
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


_plot_topomap_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}", file_name="iva_topomap_mean_var_top.png",
)
_plot_topomap_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}", file_name="iva_topomap_mean_var_bottom.png",
)

---
## Analysis (f) — Shared Time × Frequency Source Map (Direct)

Because the source *is* a `(F, T)` map, we plot the subject-averaged
source directly — **no rank-1 outer-product approximation** (contrast the
frequency_channel notebook, which had to multiply a frequency profile by a
time profile). This is the headline view: the spectro-temporal pattern
IVA found to be shared across subjects.

```
mean_sources[k] = mean over subjects of iva_sources[:, k, :, :]   # (F, T)
```

In [ ]:
# Subject-averaged spectro-temporal source: (N_PCA, F, T). Plotted directly.
mean_sources = iva_sources.mean(axis=0)  # (N_PCA, F, T)


def _plot_tf_group(indices, labels, group_name, file_name):
    n_show = len(indices)
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, k in zip(axes, labels, indices):
        data_i = mean_sources[k]  # (F, T)
        vlim_i = max(float(np.percentile(np.abs(data_i), 99)), 1e-12)
        mesh = ax.pcolormesh(
            time, FREQS, data_i, cmap="RdBu_r",
            vmin=-vlim_i, vmax=vlim_i, shading="auto",
        )
        ax.set_ylabel("Freq (Hz)")
        ax.set_title(f"{lbl} — Shared Time × Frequency source", fontsize=10)
        fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025)
        mark_stimulus_onsets(ax)
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(
        f"Per-IC Shared Time × Frequency Source (mean across subjects) — "
        f"{group_name} — {LABEL}",
        fontsize=13, y=1.01,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


_plot_tf_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}", file_name="iva_shared_time_frequency_top.png",
)
_plot_tf_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}", file_name="iva_shared_time_frequency_bottom.png",
)

---
## Analysis (g) — Source Mean & Variance Along Time and Frequency

Across-subject mean ± √variance of the two source marginals:

- **temporal**: `iva_time[s, k, :]` collapsed over subjects → vs time
- **spectral**: `iva_freq[s, k, :]` collapsed over subjects → vs frequency

Narrow bands mark where subjects agree on the source's expression; wide
bands mark idiosyncratic time points / frequency bins.

In [ ]:
def _plot_profile_group(x_vals, mean_arr, std_arr, indices, labels, *,
                        xlabel, axis_name, group_name, file_name):
    """mean ± √var profile over a 1-D axis (time or frequency)."""
    n_show = len(indices)
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.4 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for i, (ax, lbl, k) in enumerate(zip(axes, labels, indices)):
        ax.plot(x_vals, mean_arr[k], lw=0.9, color="darkorange", label="mean")
        ax.fill_between(
            x_vals, mean_arr[k] - std_arr[k], mean_arr[k] + std_arr[k],
            alpha=0.25, color="darkorange", label="± √variance",
        )
        ax.set_ylabel(lbl, fontsize=8)
        ax.set_title(f"{lbl} — {axis_name} Mean & Variance", fontsize=10)
        if "Time" in xlabel:
            mark_stimulus_onsets(ax)
        if i == 0:
            ax.legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel(xlabel)
    fig.suptitle(
        f"Per-IC {axis_name} Source — Mean & Variance — {group_name} — {LABEL}",
        fontsize=13, y=1.01,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


# Temporal profile (freq-collapsed source) across subjects.
mean_time = iva_time.mean(axis=0)  # (N_PCA, T)
std_time = iva_time.std(axis=0)
_plot_profile_group(
    time, mean_time, std_time, TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    xlabel="Time (s)", axis_name="Temporal", group_name=f"TOP {N_TOP}",
    file_name="iva_mean_variance_over_time_top.png",
)
_plot_profile_group(
    time, mean_time, std_time, BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    xlabel="Time (s)", axis_name="Temporal", group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_mean_variance_over_time_bottom.png",
)

# Spectral profile (time-collapsed source) across subjects.
mean_freq = iva_freq.mean(axis=0)  # (N_PCA, F)
std_freq = iva_freq.std(axis=0)
_plot_profile_group(
    FREQS, mean_freq, std_freq, TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    xlabel="Frequency (Hz)", axis_name="Spectral", group_name=f"TOP {N_TOP}",
    file_name="iva_mean_variance_over_freq_top.png",
)
_plot_profile_group(
    FREQS, mean_freq, std_freq, BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    xlabel="Frequency (Hz)", axis_name="Spectral", group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_mean_variance_over_freq_bottom.png",
)

---
## Analysis (h) — Pair-Space Heatmaps per Component

Per-component heatmaps over subject-involving axis pairs (**first name =
x-axis**). The third dimension is collapsed by averaging. `RdBu_r` keeps the sign in every panel (symmetric about 0). Subject-involving variants share a colour limit across components for comparability.

| Variant | Source | x | y |
|---------|--------|----|----|
| **Time × Subject** | `iva_time[:, k, :]` | time | subject |
| **Subject × Frequency** | `iva_freq[:, k, :]` | subject | frequency |
| **Subject × Channel** | `iva_components[:, k, :]` | subject | channel |

In [ ]:
channels = np.arange(n_channels)
subjects_idx = np.arange(n_subjects)
subject_labels = [f"S{s + 1}" for s in subjects_idx]


def _safe_vlim(arr: np.ndarray) -> float:
    """99th percentile of |arr|, never zero (so vmin/vmax stay valid)."""
    return max(float(np.percentile(np.abs(arr), 99)), 1e-12)


def _plot_time_subject_group(indices, labels, group_name, file_name):
    # x=time, y=subject. Source: iva_time (freq-collapsed temporal source).
    data = iva_time.transpose(1, 0, 2)  # (N_PCA, S, T)
    sel = data[indices]
    vlim = _safe_vlim(sel)
    n_show = len(indices)
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, mat in zip(axes, labels, sel):
        mesh = ax.pcolormesh(
            time, subjects_idx, mat, cmap="RdBu_r",
            vmin=-vlim, vmax=vlim, shading="auto",
        )
        ax.set_yticks(subjects_idx)
        ax.set_yticklabels(subject_labels, fontsize=8)
        ax.set_ylabel("Subject")
        ax.set_title(f"{lbl} — Time × Subject", fontsize=10)
        fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025, label="source")
        mark_stimulus_onsets(ax)
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(
        f"Time × Subject per IC (temporal marginal) — {group_name} — {LABEL}",
        fontsize=13, y=1.01,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def _plot_subject_freq_group(indices, labels, group_name, file_name):
    # x=subject, y=frequency. Source: iva_freq (time-collapsed signed source).
    data = iva_freq.transpose(1, 2, 0)  # (N_PCA, F, S)
    sel = data[indices]
    vlim = _safe_vlim(sel)
    n_show = len(indices)
    fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 5.0), sharey=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, mat in zip(axes, labels, sel):
        mesh = ax.pcolormesh(
            subjects_idx, FREQS, mat, cmap="RdBu_r",
            vmin=-vlim, vmax=vlim, shading="auto",
        )
        ax.set_xticks(subjects_idx)
        ax.set_xticklabels(subject_labels, fontsize=8)
        ax.set_xlabel("Subject")
        ax.set_title(lbl, fontsize=8)
        fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="source")
    axes[0].set_ylabel("Frequency (Hz)")
    fig.suptitle(
        f"Subject × Frequency per IC (spectral marginal) — {group_name} — {LABEL}",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def _plot_subject_chan_group(indices, labels, group_name, file_name):
    # x=subject, y=channel. Source: iva_components (channel topography).
    data = iva_components.transpose(1, 2, 0)  # (N_PCA, C, S)
    sel = data[indices]
    vlim = _safe_vlim(sel)
    n_show = len(indices)
    fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 5.0), sharey=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, mat in zip(axes, labels, sel):
        mesh = ax.pcolormesh(
            subjects_idx, channels, mat, cmap="RdBu_r",
            vmin=-vlim, vmax=vlim, shading="auto",
        )
        ax.set_xticks(subjects_idx)
        ax.set_xticklabels(subject_labels, fontsize=8)
        ax.set_xlabel("Subject")
        ax.set_title(lbl, fontsize=8)
        fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="loading")
    axes[0].set_ylabel("Channel")
    fig.suptitle(
        f"Subject × Channel per IC (topographies) — {group_name} — {LABEL}",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


# ── (1) Time × Subject ──────
_plot_time_subject_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}", file_name="iva_pairmap_time_subject_top.png",
)
_plot_time_subject_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}", file_name="iva_pairmap_time_subject_bottom.png",
)

# ── (2) Subject × Frequency ──────
_plot_subject_freq_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}", file_name="iva_pairmap_subject_frequency_top.png",
)
_plot_subject_freq_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_pairmap_subject_frequency_bottom.png",
)

# ── (3) Subject × Channel ──────
_plot_subject_chan_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}", file_name="iva_pairmap_subject_channel_top.png",
)
_plot_subject_chan_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_pairmap_subject_channel_bottom.png",
)

---
## Analysis (i) — Mean Subject Loading per Component

Mean absolute spectro-temporal score per subject — how strongly each
participant expresses the kth source over the whole `F·T` axis:

```
subject_loadings[s, k] = mean over (f, t) of |iva_scores_pca[s, k, :]|
```

In [ ]:
# Per-subject mean |spectro-temporal score|: (S, N_PCA)
subject_loadings = np.abs(iva_scores_pca).mean(axis=2)  # (S, K)


def _plot_loadings_group(indices, labels, group_name, file_name):
    n_show = len(indices)
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, k in zip(axes, labels, indices):
        ax.barh(range(n_subjects), subject_loadings[:, k], color="darkorange")
        ax.set_yticks(range(n_subjects))
        ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
        ax.set_xlabel("|score|")
        ax.set_title(lbl, fontsize=8)
    axes[0].set_ylabel("Subject")
    fig.suptitle(
        f"Per-Subject Mean Loading per Component — {group_name} — {LABEL}",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


_plot_loadings_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}", file_name="iva_subject_loadings_top.png",
)
_plot_loadings_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}", file_name="iva_subject_loadings_bottom.png",
)